In [ ]:
# ── EXP-03: U-Net + Focal Loss + Data Augmentation ───────────────────────────
data_dir    = 'xBD_UC3M'
patch_size  = 128
num_epochs  = 30
batch_size  = 4
num_workers = 0

BASE_RESULT_DIR = r'/Users/juan.macias@feverup.com/Desktop/cv/cv-2a-image-segmentation/VA_Pr2A_ImageSegmentation_2025_2026/results/parte-4'
result_dir      = os.path.join(BASE_RESULT_DIR, 'exp03_unet_focal_augmentation')
model_name      = 'unet_focal_aug_exp03'

num_classes = 2
class_names = ['background', 'building']

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

# ── Dataset statistics (cached from EXP-01) ───────────────────────────────────
stats_cache = os.path.join(BASE_RESULT_DIR, 'xbd_stats.npy')
image_stats = np.load(stats_cache, allow_pickle=True).item()
print(f"Stats loaded — mean={image_stats['mean']}  std={image_stats['std']}")

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds_raw = xBDDataset(data_dir, split=['train'], task='segmentation',
                           patch_size=patch_size, stats=image_stats, transform=None)
val_ds_raw   = xBDDataset(data_dir, split=['val'],   task='segmentation',
                           patch_size=patch_size, stats=image_stats, transform=None)
test_ds_raw  = xBDDataset(data_dir, split=['test'],  task='segmentation',
                           patch_size=patch_size, stats=image_stats, transform=None)


# ── KeyAdapter with optional augmentation ─────────────────────────────────────
# Geometric transforms (flip, rot90) are applied identically to image and mask.
# Photometric transforms (brightness, contrast) are applied only to the image.
# Images are normalized to ~[-1, 1], so we work directly on tensors.
class KeyAdapterBinaryAug(Dataset):
    def __init__(self, ds, augment=False):
        self.ds      = ds
        self.augment = augment

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        s     = self.ds[idx]
        image = s['patch_post'].clone()                    # (3, H, W)
        mask  = (s['mask_patch_raw'] != 255).long()        # (H, W)

        if self.augment:
            # ── Geometric (image + mask) ──────────────────────────────────────
            if random.random() > 0.5:                      # horizontal flip
                image = torch.flip(image, [2])
                mask  = torch.flip(mask,  [1])
            if random.random() > 0.5:                      # vertical flip
                image = torch.flip(image, [1])
                mask  = torch.flip(mask,  [0])
            k = random.randint(0, 3)                       # 0/90/180/270 rotation
            if k:
                image = torch.rot90(image, k, [1, 2])
                mask  = torch.rot90(mask,  k, [0, 1])

            # ── Photometric (image only) ──────────────────────────────────────
            # brightness: additive shift on normalized tensors
            image = torch.clamp(image + random.uniform(-0.1, 0.1), -1.0, 1.0)
            # contrast: scale around channel mean
            for c in range(image.shape[0]):
                mu = image[c].mean()
                image[c] = torch.clamp((image[c] - mu) * random.uniform(0.8, 1.2) + mu, -1.0, 1.0)

        return {'image': image, 'mask': mask, 'img_path': s['patch_post_path']}


dataloaders = {
    'Train': DataLoader(KeyAdapterBinaryAug(train_ds_raw, augment=True),  batch_size=batch_size,
                        shuffle=True,  num_workers=num_workers),
    'Val':   DataLoader(KeyAdapterBinaryAug(val_ds_raw,   augment=False), batch_size=1,
                        shuffle=False, num_workers=num_workers),
    'Test':  DataLoader(KeyAdapterBinaryAug(test_ds_raw,  augment=False), batch_size=1,
                        shuffle=False, num_workers=num_workers),
}


# ── Focal Loss ────────────────────────────────────────────────────────────────
# Focal Loss = (1 - p_t)^gamma * CE(input, target)
# p_t = exp(-CE) is the model's confidence on the correct class.
# When p_t is high (easy pixel) the factor ~0 → suppressed gradient.
# When p_t is low (hard pixel)  the factor ~1 → full gradient.
# alpha weights address class imbalance (bg/building ratio = 4.6x from training set).
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # class-weight tensor, same role as in CrossEntropyLoss

    def forward(self, inputs, targets):
        # per-pixel CE (no reduction), with optional class weights
        ce     = torch.nn.functional.cross_entropy(inputs, targets,
                                                   weight=self.alpha, reduction='none')
        p_t    = torch.exp(-ce)                    # confidence on correct class
        focal  = (1.0 - p_t) ** self.gamma * ce
        return focal.mean()


# ── U-Net (same architecture as EXP-02) ───────────────────────────────────────
class UNetBinary(torch.nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.pool = torch.nn.MaxPool2d(2)
        self.enc1 = self._block(3,    32)
        self.enc2 = self._block(32,   64)
        self.enc3 = self._block(64,  128)
        self.btn  = self._block(128, 256)
        self.up   = torch.nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3 = self._block(256+128, 128)
        self.dec2 = self._block(128+ 64,  64)
        self.dec1 = self._block( 64+ 32,  32)
        self.head = torch.nn.Conv2d(32, num_classes, kernel_size=1)

    @staticmethod
    def _block(in_ch, out_ch):
        return torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            torch.nn.ReLU(inplace=True),
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.btn(self.pool(e3))
        d  = self.dec3(torch.cat([self.up(b),  e3], dim=1))
        d  = self.dec2(torch.cat([self.up(d),  e2], dim=1))
        d  = self.dec1(torch.cat([self.up(d),  e1], dim=1))
        return {'out': self.head(d)}


In [ ]:
model = UNetBinary(num_classes).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# ── Focal Loss with alpha from measured pixel ratio (bg/building = 4.6x) ──────
alpha     = torch.tensor([1.0, 4.6]).to(device)
criterion = FocalLoss(gamma=2.0, alpha=alpha)

# ── Optimiser + cosine LR schedule ───────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
lr_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
metrics   = {'jaccard_score': jaccard_score}

os.makedirs(result_dir, exist_ok=True)

# ── Train ─────────────────────────────────────────────────────────────────────
model = train_model(
    model, criterion, dataloaders, device,
    optimizer, lr_sched, metrics,
    result_dir, model_name,
    num_classes=num_classes, num_epochs=num_epochs,
)

# ── Evaluate on test set ──────────────────────────────────────────────────────
weights = torch.load(os.path.join(result_dir, model_name + '_best.pth.tar'))['state_dict']
model.to(device)
model.load_state_dict(weights)
model.eval()
cm_total = test_segmentation_model(model, dataloaders, num_classes, class_names, result_dir, True, batchsize_test)
